In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CSV_FILE = DATA_DIR / "anes_specialstudy_2020-2022_socialmedia_csv_20230705.csv"

OUT_DIR = DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
SAMPLE_SIZES = [100]
SEED = 42

STRATA_COLS = [
    "profile_gender",
    "profile_age_bucket",
    "profile_racethnicity",
    "profile_educ5"
]

CHECK_COLS = [
    "profile_gender",
    "profile_age_bucket",
    "profile_racethnicity",
    "profile_educ5",
    "profile_region4",
    "profile_relig",
    "w2presvtwho"
]

In [ ]:
# =====================================================
# LOAD + CLEAN ANES
# =====================================================

def get_personas(csv_file):
    anes = pd.read_csv(csv_file, encoding="mac_roman", low_memory=False)

    # Remove null weights and convert
    anes = anes[anes["weight_post"] != "#NULL!"].reset_index(drop=True)
    anes["weight_post"] = anes["weight_post"].astype(float)

    # Keep only respondents with valid numeric nd_fd and nd_fr
    anes["nd_fd"] = pd.to_numeric(anes["nd_fd"], errors="coerce")
    anes["nd_fr"] = pd.to_numeric(anes["nd_fr"], errors="coerce")

    anes = anes[
        anes["nd_fd"].notna()
        & anes["nd_fr"].notna()
        & ((anes["nd_fd"] + anes["nd_fr"]) <= 6)
    ].reset_index(drop=True)

    # Gender
    anes["profile_gender"] = anes["profile_gender"].replace({
        1: "male",
        2: "female"
    })

    # Race/ethnicity
    anes["profile_racethnicity"] = anes["profile_racethnicity"].replace({
        1: "white",
        2: "black",
        3: "ethnic minority",
        4: "hispanic"
    })

    # Education
    anes["profile_educ5"] = anes["profile_educ5"].replace({
        1: "less than high school",
        2: "high school graduate",
        3: "vocational tech school",
        4: "bachelor's degree",
        5: "postgraduate"
    })

    # Religion
    anes["profile_relig"] = anes["profile_relig"].replace({
        1: "Protestant",
        2: "Roman Catholic",
        9: "atheist",
        10: "agnostic",
        11: "not religious",
        12: "just Christian",
        14: "believer in a minority religion",
        -7: "not religious",
        -1: "not religious"
    })

    # Political vote
    anes["w2presvtwho"] = anes["w2presvtwho"].replace({
        1: "Republican",
        2: "Democrat",
        3: "Independent",
        -7: "didn't vote",
        -1: "didn't vote"
    })

    # Age
    anes["profile_age"] = pd.to_numeric(anes["profile_age"], errors="coerce")

    anes["profile_age_bucket"] = pd.cut(
        anes["profile_age"],
        bins=[17, 29, 44, 59, 74, 120],
        labels=["18-29", "30-44", "45-59", "60-74", "75+"]
    ).astype(str)

    # Region
    anes["profile_region4"] = anes["profile_region4"].replace({
        1: "Northeast",
        2: "Midwest",
        3: "South",
        4: "West"
    })

    required = [
        "caseid",
        "weight_post",
        "nd_fd",
        "nd_fr",
        "profile_gender",
        "profile_age",
        "profile_age_bucket",
        "profile_racethnicity",
        "profile_educ5",
        "profile_state",
        "profile_region4",
        "profile_relig",
        "w2presvtwho"
    ]

    anes = anes[required].dropna().reset_index(drop=True)

    return anes

In [ ]:
# =====================================================
# STRATIFIED WEIGHTED SAMPLING
# =====================================================

def compute_strata_allocation(df, strata_cols, weight_col, n):
    strata = (
        df.groupby(strata_cols)[weight_col]
        .sum()
        .reset_index(name="w_sum")
    )

    total_weight = strata["w_sum"].sum()

    strata["raw_target_n"] = strata["w_sum"] / total_weight * n
    strata["target_n"] = np.floor(strata["raw_target_n"]).astype(int)

    remainder = n - strata["target_n"].sum()

    if remainder > 0:
        frac = strata["raw_target_n"] - strata["target_n"]
        add_idx = frac.nlargest(remainder).index
        strata.loc[add_idx, "target_n"] += 1

    return strata


def stratified_weighted_sample(
    df,
    strata_cols,
    weight_col,
    n,
    seed=42
):
    rng = np.random.default_rng(seed)

    allocation = compute_strata_allocation(
        df=df,
        strata_cols=strata_cols,
        weight_col=weight_col,
        n=n
    )

    samples = []

    for _, row in allocation.iterrows():
        mask = np.ones(len(df), dtype=bool)

        for col in strata_cols:
            mask &= df[col].eq(row[col])

        subset = df.loc[mask]

        k = int(row["target_n"])

        if k == 0 or len(subset) == 0:
            continue

        weights = subset[weight_col].to_numpy()
        probs = weights / weights.sum()

        replace = len(subset) < k

        sampled_idx = rng.choice(
            subset.index,
            size=k,
            replace=replace,
            p=probs
        )

        samples.append(subset.loc[sampled_idx])

    sampled = pd.concat(samples).reset_index(drop=True)

    # Safety check: ensure exact requested size
    if len(sampled) != n:
        raise ValueError(
            f"Sample size mismatch: expected {n}, got {len(sampled)}"
        )

    return sampled

In [ ]:
# =====================================================
# DIAGNOSTICS
# =====================================================

def weighted_proportions(df, col, weight_col="weight_post"):
    out = df.groupby(col)[weight_col].sum()
    return out / out.sum()


def unweighted_proportions(df, col):
    return df[col].value_counts(normalize=True).sort_index()


def compare_sample_to_full(full_df, sample_df, col, weight_col="weight_post"):
    full = weighted_proportions(full_df, col, weight_col)
    sample = unweighted_proportions(sample_df, col)

    out = pd.concat(
        [full, sample],
        axis=1,
        keys=["full_weighted", "sample_unweighted"]
    ).fillna(0)

    out["abs_diff"] = (
        out["sample_unweighted"] - out["full_weighted"]
    ).abs()

    return out.sort_index()


def print_balance_report(full_df, sample_df, sample_name, cols):
    print(f"\n==================== {sample_name} ====================")

    for col in cols:
        print(f"\n--- {col} ---")
        tab = compare_sample_to_full(full_df, sample_df, col)
        print(tab.round(3))

In [ ]:
# =====================================================
# RUN
# =====================================================

df = get_personas(CSV_FILE)

print(f"Clean sample: {len(df)} respondents")
print(df.head())

samples = {}

for n in SAMPLE_SIZES:
    sample = stratified_weighted_sample(
        df=df,
        strata_cols=STRATA_COLS,
        weight_col="weight_post",
        n=n,
        seed=SEED + n
    )

    samples[n] = sample

    out_path = os.path.join(OUT_DIR, f"anes_{n}.csv")
    sample.to_csv(out_path, index=False)

    print(f"\nSaved {n}-person sample to:")
    print(out_path)

    print_balance_report(
        full_df=df,
        sample_df=sample,
        sample_name=f"N={n}",
        cols=CHECK_COLS
    )